In [1]:
import os
import arcpy

output_location = r"\\metanoia\GeoData\DepartmentOfUtilities\Simpson_A\gdbs"
GDB_name = "middlesexparcelanalysis.gdb"

PARCELS_URL = "https://services9.arcgis.com/UrElc2PRESEUr5jw/arcgis/rest/services/Parcels/FeatureServer/14"
FLOOD_URL = "https://services5.arcgis.com/7weheFjxuNkGGiZi/arcgis/rest/services/USA_Flood_Hazard_Areas_view/FeatureServer/0"

FLOOD_WHERE = "DFIRM_ID LIKE '51119%'"
RES_WHERE = "Zone LIKE 'R%'"
NONRES_WHERE = "Zone NOT LIKE 'R%' OR Zone IS NULL"
RES_BUFFER = "100 Feet"

gdb = os.path.join(output_location, GDB_name)
if not arcpy.Exists(gdb):
    arcpy.management.CreateFileGDB(output_location, GDB_name)
arcpy.env.workspace = gdb
arcpy.env.overwriteOutput = True

parcel_sr = arcpy.Describe(PARCELS_URL).spatialReference
flood_sr = arcpy.Describe(FLOOD_URL).spatialReference
print(f"Parcel CRS: {parcel_sr.name} (WKID {parcel_sr.factoryCode})")
print(f"FEMA CRS:   {flood_sr.name} (WKID {flood_sr.factoryCode})")

zones = {r[0] for r in arcpy.da.SearchCursor(PARCELS_URL, ["Zone"])}
print("Zone values:", sorted(zones, key=lambda v: (v is None, str(v))))

# get FEMA data for only 51119 and reproject to 2284
env = {"outputCoordinateSystem": parcel_sr}
transforms = arcpy.ListTransformations(flood_sr, parcel_sr)
if transforms:
    env["geographicTransformations"] = transforms[0]
    print("Datum transformation:", transforms[0])

print("Exporting FEMA flood zones for 51119...")
with arcpy.EnvManager(**env):
    arcpy.conversion.ExportFeatures(FLOOD_URL, "fema_51119", where_clause=FLOOD_WHERE)
print("  polygons:", arcpy.management.GetCount("fema_51119")[0])

# deal with parcels
print("Exporting parcels...")
arcpy.conversion.ExportFeatures(PARCELS_URL, "parcels_res", where_clause=RES_WHERE)
arcpy.conversion.ExportFeatures(PARCELS_URL, "parcels_nonres_all", where_clause=NONRES_WHERE)
print("  R parcels:    ", arcpy.management.GetCount("parcels_res")[0])
print("  non-R parcels:", arcpy.management.GetCount("parcels_nonres_all")[0])

# grab only non res and more than 100 feet from res
lyr = arcpy.management.MakeFeatureLayer("parcels_nonres_all", "nonres_lyr")
arcpy.management.SelectLayerByLocation(
    lyr, "WITHIN_A_DISTANCE", "parcels_res", RES_BUFFER, "NEW_SELECTION", "INVERT")
arcpy.conversion.ExportFeatures(lyr, "parcels_nonres")
arcpy.management.Delete(lyr)
arcpy.management.Delete("parcels_nonres_all")

print(f"  non-R parcels > {RES_BUFFER} from R:", arcpy.management.GetCount("parcels_nonres")[0])
print("Done:", gdb)

Parcel CRS: NAD_1983_StatePlane_Virginia_South_FIPS_4502_Feet (WKID 2284)
FEMA CRS:   WGS_1984_Web_Mercator_Auxiliary_Sphere (WKID 3857)
Zone values: ['', '99', 'C', 'CD', 'DRC', 'GB', 'GBVC', 'H', 'LDR', 'LI', 'LRD', 'MHD', 'R', 'RH', 'VC', 'WC']
Datum transformation: WGS_1984_(ITRF00)_To_NAD_1983
Exporting FEMA flood zones for 51119...
  polygons: 652
Exporting parcels...
  R parcels:     5016
  non-R parcels: 9007
  non-R parcels > 100 Feet from R: 7260
Done: \\metanoia\GeoData\DepartmentOfUtilities\Simpson_A\gdbs\middlesexparcelanalysis.gdb


In [2]:
FLOODPLAIN_WHERE = "SFHA_TF = 'T'"   # 1% annual chance
SETBACK = "-20 Feet"              

print("Removing parcels that touch the floodplain...")
sfha_lyr = arcpy.management.MakeFeatureLayer("fema_51119", "sfha_lyr", FLOODPLAIN_WHERE)
lyr = arcpy.management.MakeFeatureLayer("parcels_nonres", "nonres_lyr")
arcpy.management.SelectLayerByLocation(
    lyr, "INTERSECT", sfha_lyr, None, "NEW_SELECTION", "INVERT")
arcpy.conversion.ExportFeatures(lyr, "parcels_dry")
arcpy.management.Delete(lyr)
arcpy.management.Delete(sfha_lyr)
print("  parcels clear of floodplain:", arcpy.management.GetCount("parcels_dry")[0])

print(f"Buffering {SETBACK} for buildable area...")
arcpy.analysis.Buffer("parcels_dry", "parcels_buildable", SETBACK, "FULL", "ROUND", "NONE")

# if buffer removes all area, delete parcel
dropped = 0
with arcpy.da.UpdateCursor("parcels_buildable", ["SHAPE@"]) as cur:
    for (shp,) in cur:
        if shp is None or shp.area == 0:
            cur.deleteRow()
            dropped += 1

arcpy.management.AddField("parcels_buildable", "BUILD_AC", "DOUBLE")
arcpy.management.CalculateField("parcels_buildable", "BUILD_AC", "!shape.area@acres!", "PYTHON3")

total_ac = sum(r[0] for r in arcpy.da.SearchCursor("parcels_buildable", ["BUILD_AC"]))
print(f"  dropped {dropped} parcels too narrow for the setback")
print("  buildable parcels:", arcpy.management.GetCount("parcels_buildable")[0])
print(f"  total buildable acres: {total_ac:,.1f}")

Removing parcels that touch the floodplain...
  parcels clear of floodplain: 5200
Buffering -20 Feet for buildable area...
  dropped 0 parcels too narrow for the setback
  buildable parcels: 4766
  total buildable acres: 22,163.9
